# EXP-2026-001 on Colab: representation fallback gates

The reference gate (256 px, gray-level mask codec) failed. This notebook gates the three preregistered fallbacks on the same 24 accepted training patients. Each variant writes its own report; none overwrites another.

| Variant | Working size | Mask encoding |
|---|---|---|
| `512_scalar` | 512 px, same field of view (0.5 mm/px) | reference gray levels |
| `256_class_rgb` | 256 px (1 mm/px) | one class per color channel |
| `512_class_rgb` | 512 px, same field of view | one class per color channel |

Selection rule, fixed before any of these ran: the first variant in this order that passes on every path is selected; if none passes, it is a representation failure.

Before running: copy the laptop's updated `data/RHUH-GBM/synthesis/EXP-2026-001/test_snapshots/` folder to the same place on Drive (a new tests archive plus `LATEST.txt`). Use the GPU runtime.

**Clear all outputs before saving, sharing or committing this notebook.**

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"  # your fork only
BRANCH = "exp-2026-001-implementation"
DRIVE_RHUH = "/content/drive/MyDrive/RHUH-GBM"
WORK = "/content/MedSegFactory"

## 1. Mount Drive, get the latest code from the fork, link the data

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
if not Path(WORK).is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
link = Path(WORK, "data", "RHUH-GBM")
link.parent.mkdir(exist_ok=True)
if not link.exists():
    link.symlink_to(DRIVE_RHUH, target_is_directory=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
PRIVATE = Path("data/RHUH-GBM/synthesis/EXP-2026-001")
assert (PRIVATE / "split/split.json").exists(), "The synthesis folder is missing on Drive"
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

## 2. Dependencies and tests

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "diffusers==0.40.0", "transformers==5.17.0", "accelerate==1.15.0",
                "nibabel>=5.3,<6", "SimpleITK==2.5.6", "pydicom>=3,<4", "scikit-image>=0.25"], check=True)

import tarfile

snapshots = PRIVATE / "test_snapshots"
name = (snapshots / "LATEST.txt").read_text(encoding="utf-8").strip()
with tarfile.open(snapshots / name) as archive:
    try:
        archive.extractall(WORK, filter="data")
    except TypeError:
        archive.extractall(WORK)
result = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print("tests snapshot:", name)
print(result.stdout[-1500:])
print(result.stderr[-1000:])

## 3. Run the three fallback gates (GPU; the 512 px variants take longer)

In [ ]:
ORDER = [
    ("512_scalar", "scalar_id_over_max_rgb", 2),
    ("256_class_rgb", "class_separated_rgb", 1),
    ("512_class_rgb", "class_separated_rgb", 2),
]
policy = json.loads((PRIVATE / "registration_policy/acceptance_policy_v1.json").read_text(encoding="utf-8"))
for name, codec, upsample in ORDER:
    report = PRIVATE / f"representation_gate_{name}" / "report.json"
    if report.exists():
        print(f"{name}: report already exists, not rerun (write-once)")
        continue
    command = [sys.executable, "scripts/representation_gate.py", "--allow-download", "--device", "cuda",
               "--codec", codec, "--upsample", str(upsample), "--batch", "8" if upsample == 2 else "16",
               "--exclude", *policy["excluded_training"]]
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout[-1500:])
    if result.returncode:
        print(result.stderr[-3000:])

## 4. Compare all variants and apply the selection rule

In [ ]:
import pandas as pd

folders = {"256_scalar (reference, v1)": "representation_gate_v1"}
folders.update({name: f"representation_gate_{name}" for name, _, _ in ORDER})
rows, passed = [], {}
for label, folder in folders.items():
    path = PRIVATE / folder / "report.json"
    if not path.exists():
        print("missing:", label)
        continue
    report = json.loads(path.read_text(encoding="utf-8"))
    passed[label] = report["pass"]
    for path_name, s in report["paths"].items():
        rows.append({
            "variant": label, "path": path_name,
            "id3_dice": round(s["macro_id3_dice"], 3), "id3_recall": round(s["macro_id3_recall"], 3),
            "lesions_kept": f"{s['pooled_matched_lesions']}/{s['pooled_reference_lesions']}",
            "fake_id3_ml": round(s["mean_spurious_id3_ml_negative"], 4),
            "fake_id1_ml": round(s["mean_unsupported_id1_ml"], 4), "pass": s["pass"],
        })
selected = next((name for name, _, _ in ORDER if passed.get(name)), None)
print("Selected variant:", selected or "none: representation failure")
pd.DataFrame(rows)

## Next

Send Claude the "Selected variant" line and the table (aggregate numbers only). Training code then switches to the selected representation, and the gate is rerun after any later change.